# 1.6 — SE(3) Frame Diffusion

**Mechanism of the day:** generative modelling where the data does not live in a flat
vector space. Take the flow matching of 1.4 and run it on the manifold of **rigid
motions**, so the thing you generate is a protein **backbone**.

This is the last generative core, and it is the one that made structure design work.
Back in **0.1** you built the object: a per-residue **frame** — a rotation `R` and a
translation `t` saying *"this residue sits here, oriented like this."* RFdiffusion,
FrameFlow, FoldFlow and Chroma all generate exactly that.

**Why not just diffuse the 9 numbers in `R`?** Because rotation matrices are not free
parameters. They must satisfy `R R^T = I` and `det R = +1`. Add Gaussian noise to a
rotation matrix and you get a matrix that is no longer a rotation — it shears and
scales. Every step of a naive sampler drifts further off the manifold, and there is no
"project back" that is not itself a modelling decision. We will demonstrate this
failure in two lines before fixing it.

**The fix** is to do all the arithmetic in the **tangent space**. For rotations:

- `so3_exp(w)` turns a 3-vector `w` (axis * angle) into a rotation — always a valid one.
- `so3_log(R)` goes back.
- The geodesic from `R_0` to `R_1` is `R_0 exp(t * log(R_0^T R_1))` — the shortest path
  *along the manifold*, the curved analogue of 1.4's straight line.

And here is the pleasant surprise: differentiate that geodesic and the body-frame
angular velocity is **constant**, `w = log(R_0^T R_1)` — exactly as `x1 - x0` was
constant in 1.4. So the flow-matching recipe transfers unchanged:

```
rotations:     regress  w = log(R_data^T R_noise)     integrate with  R <- R exp(-dt * w)
translations:  regress  v = x_noise - x_data          integrate with  x <- x - dt * v
```

Same objective, same ODE, one replaced with its curved counterpart.

**Our data:** short backbones built by composing one fixed screw motion per residue —
which is precisely what secondary structure *is*. Two classes, an **alpha-helix**
(~100 deg twist per residue) and a **beta-strand** (~180 deg), plus jitter. The model
has to produce backbones that commit to one or the other, and we can check that with a
single number per backbone.

**How to use this notebook:** implement the reps, make the checkpoints pass. Solutions
at the bottom. Trains in ~30s on a laptop CPU.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
rng = np.random.default_rng(0)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK = '#2a78d6', '#008300', '#52514e'
L = 12                                   # residues per backbone

def hat(w):
    '''3-vector -> skew-symmetric matrix (the so(3) Lie algebra element).'''
    O = torch.zeros(*w.shape[:-1], 3, 3, dtype=w.dtype)
    O[..., 0, 1] = -w[..., 2]; O[..., 0, 2] =  w[..., 1]
    O[..., 1, 0] =  w[..., 2]; O[..., 1, 2] = -w[..., 0]
    O[..., 2, 0] = -w[..., 1]; O[..., 2, 1] =  w[..., 0]
    return O

def rand_rot(n):
    '''Uniformly random rotations — our noise distribution on SO(3).'''
    A = torch.randn(n, 3, 3)
    Q, S = torch.linalg.qr(A)
    Q = Q * torch.sign(torch.diagonal(S, dim1=-2, dim2=-1))[:, None, :]
    Q[torch.det(Q) < 0, :, 0] *= -1
    return Q

print('L =', L, 'residues | a frame is R [3,3] plus t [3]')

## Part 1 — why you cannot just add noise to a rotation

Before building anything, let us watch the naive approach break. Take a valid rotation,
add a little Gaussian noise to its nine entries, and check whether it is still a
rotation.

In [ ]:
R = rand_rot(1)[0]
noisy = R + 0.05 * torch.randn(3, 3)
print('valid rotation:  ||R R^T - I|| = %.2e   det = %+.4f' %
      ((R @ R.T - torch.eye(3)).abs().max(), torch.det(R)))
print('after adding noise to the 9 entries:')
print('                 ||R R^T - I|| = %.2e   det = %+.4f' %
      ((noisy @ noisy.T - torch.eye(3)).abs().max(), torch.det(noisy)))
print('\nThat matrix is no longer a rotation: it shears and rescales space.')
print('Doing this a few hundred times (as a sampler would) drifts badly off SO(3).')

### Rep 1 — `so3_exp(w)`
Rodrigues' formula. `w` is `[..., 3]` — an axis scaled by an angle — and the output is
`[..., 3, 3]`, **always** a valid rotation no matter what you feed it. That is the
whole point: the exponential map is a chart from an unconstrained vector space onto the
manifold.

```
theta = |w| ;  K = hat(w / theta)
R = I + sin(theta) K + (1 - cos(theta)) K^2
```
Guard the division with `.clamp_min(1e-8)` so `w = 0` gives the identity.

In [ ]:
def so3_exp(w):
    '''Rotation vector [..., 3] -> rotation matrix [..., 3, 3].'''
    # YOUR CODE HERE
    # hint: th = w.norm(dim=-1, keepdim=True).clamp_min(1e-8); K = hat(w / th)
    #       then th = th[..., None] so it broadcasts against the [3, 3] block
    raise NotImplementedError

# --- checkpoint ---
w = torch.randn(200, 3) * 0.4
Rw = so3_exp(w)
assert Rw.shape == (200, 3, 3)
assert (Rw @ Rw.transpose(-1, -2) - torch.eye(3)).abs().max() < 1e-5, 'must be orthonormal'
assert (torch.det(Rw) - 1).abs().max() < 1e-5, 'must be a PROPER rotation (det +1)'
assert (so3_exp(torch.zeros(1, 3))[0] - torch.eye(3)).abs().max() < 1e-6, 'w=0 -> identity'
# a rotation of pi about z flips x and y
half = so3_exp(torch.tensor([[0.0, 0.0, math.pi]]))[0]
assert torch.allclose(half @ torch.tensor([1.0, 0, 0]), torch.tensor([-1.0, 0, 0]), atol=1e-5)
print('so3_exp ✓ — every output is a genuine rotation, for ANY input vector')

### Rep 2 — `so3_log(R)`
The inverse map, `SO(3) -> R^3`:

```
theta = arccos( (trace(R) - 1) / 2 )
w = theta / (2 sin theta) * [ R32 - R23,  R13 - R31,  R21 - R12 ]
```
Clamp the `arccos` argument into `(-1, 1)` and guard `sin(theta)`, or you will get
`NaN`s the moment a rotation is near the identity or near a half-turn.

In [ ]:
def so3_log(R):
    '''Rotation matrix [..., 3, 3] -> rotation vector [..., 3].'''
    # YOUR CODE HERE
    # hint: tr = R[..., 0, 0] + R[..., 1, 1] + R[..., 2, 2]
    #       cos = ((tr - 1) / 2).clamp(-1 + 1e-7, 1 - 1e-7); th = torch.acos(cos)
    raise NotImplementedError

# --- checkpoint ---
axis = F.normalize(torch.randn(500, 3), dim=-1)
ang = torch.rand(500, 1) * 3.0 + 0.01           # keep |w| < pi, where log is the true inverse
w = axis * ang
assert (so3_log(so3_exp(w)) - w).abs().max() < 1e-3, 'log must invert exp'
assert so3_log(torch.eye(3)[None]).abs().max() < 1e-5, 'log(I) = 0'
assert not torch.isnan(so3_log(so3_exp(torch.zeros(4, 3)))).any(), 'must not NaN at identity'
print('so3_log ✓ — round-trip error %.2e' % (so3_log(so3_exp(w)) - w).abs().max())

## Part 2 — toy backbones

A backbone is built by applying the **same rigid step** over and over: rotate a little,
translate a little, place the next residue. The size of that step is what distinguishes
secondary structure — a helix turns ~100 deg per residue and rises slowly; a strand
turns ~180 deg and extends much further.

We centre each backbone's translations and scale them to roughly unit size, exactly as
we normalized the data in 1.3. Real methods must also handle global rotation/translation
invariance properly; we keep our data in a canonical orientation so the notebook stays
about the manifold, not about frame alignment.

In [ ]:
# Rotation is about the body z-axis, so z ACCUMULATES (the rise) while the x
# component sweeps around it (the radius). These give real geometry: CA-CA 3.8 A,
# helix radius ~2.7 A rising 1.5 A/residue; strand extending 3.3 A/residue.
CLASSES = {'helix': (100.0, [3.5, 0.0, 1.5]), 'strand': (180.0, [1.0, 0.0, 3.3])}

def make_backbones(n, jitter_deg=4.0, jitter_t=0.12):
    Rs = torch.zeros(n, L, 3, 3); Ts = torch.zeros(n, L, 3)
    which = rng.integers(0, 2, size=n)
    for i in range(n):
        th, d = CLASSES[list(CLASSES)[which[i]]]
        R, t = torch.eye(3), torch.zeros(3)
        for j in range(L):
            Rs[i, j], Ts[i, j] = R, t
            ax = torch.tensor([0.0, 0.0, 1.0]) * math.radians(th)
            ax = ax + torch.randn(3) * math.radians(jitter_deg)
            R = R @ so3_exp(ax[None])[0]
            t = t + R @ (torch.tensor(d) + torch.randn(3) * jitter_t)
    Ts = Ts - Ts.mean(1, keepdim=True)
    return Rs, Ts / 10.0, which

t0 = time.time()
R_data, X_data, lab = make_backbones(3000)
print('built', tuple(R_data.shape), 'frames in %.1fs' % (time.time() - t0))

def show3d(ax, x, title, color=BLUE):
    p = x.numpy()
    ax.plot(p[:, 0], p[:, 1], p[:, 2], '-o', ms=3, lw=1.6, color=color)
    ax.set_title(title, fontsize=10)
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    # EQUAL aspect on all three axes — otherwise matplotlib rescales each one
    # independently and a long thin helix gets squashed into an unreadable tangle.
    c, r = p.mean(0), np.abs(p - p.mean(0)).max()
    ax.set_xlim(c[0] - r, c[0] + r)
    ax.set_ylim(c[1] - r, c[1] + r)
    ax.set_zlim(c[2] - r, c[2] + r)

fig = plt.figure(figsize=(9, 3.4))
for k, (name, idx) in enumerate([('helix', np.where(lab == 0)[0][0]),
                                 ('strand', np.where(lab == 1)[0][0])]):
    ax = fig.add_subplot(1, 2, k + 1, projection='3d')
    show3d(ax, X_data[idx], 'real ' + name, [BLUE, GREEN][k])
plt.tight_layout(); plt.show()

### Rep 3 — `twist_angles(Rs)`
Our evaluation metric, and the thing that defines the two classes. For each consecutive
pair of frames, the **relative** rotation is `R_i^T R_{i+1}`; its rotation angle is the
norm of its log. Return the angles in **degrees**, shape `[n, L-1]`.

In [ ]:
def twist_angles(Rs):
    '''Per-residue relative rotation angle, in degrees. [n, L, 3, 3] -> [n, L-1].'''
    # YOUR CODE HERE
    # hint: rel = Rs[:, :-1].transpose(-1, -2) @ Rs[:, 1:]
    #       angle = so3_log(rel).norm(dim=-1), then convert radians -> degrees
    raise NotImplementedError

# --- checkpoint ---
ta = twist_angles(R_data)
assert ta.shape == (len(R_data), L - 1)
h_mean, s_mean = ta[lab == 0].mean().item(), ta[lab == 1].mean().item()
assert 90 < h_mean < 110, 'helices should twist ~100 deg per residue'
assert 165 < s_mean < 185, 'strands should twist ~180 deg per residue'
print('real twist per residue:  helix %.1f deg  |  strand %.1f deg ✓' % (h_mean, s_mean))
print('(strands read slightly under 180 because a half-turn sits at the edge of')
print(' the log map, where +theta and -theta are the same rotation.)')

## Part 3 — flow matching on the manifold

Now the 1.4 recipe, twice: once on `R^3` for translations (unchanged), once on `SO(3)`
for rotations (geodesic instead of straight line).

```
w  = log(R_data^T R_noise)              # constant body-frame angular velocity
R_t = R_data @ exp(t * w)               # the geodesic;  t=0 -> data, t=1 -> noise
v  = x_noise - x_data                   # exactly as in 1.4
x_t = (1 - t) x_data + t x_noise
```

The network sees the frames at time `t` and predicts `(w, v)` per residue. It is a small
bidirectional transformer over the `L` residues — every residue needs to see the others,
because "am I in a helix or a strand" is a property of the whole chain.

In [ ]:
class FrameNet(nn.Module):
    '''Frames at time t -> (angular velocity, linear velocity) per residue.'''
    def __init__(self, d=128, nh=4, nl=3):
        super().__init__()
        self.inp = nn.Linear(12, d)              # 9 rotation entries + 3 translation
        self.pos = nn.Embedding(L, d)
        self.tmlp = nn.Sequential(nn.Linear(1, d), nn.SiLU(), nn.Linear(d, d))
        enc = nn.TransformerEncoderLayer(d, nh, 4 * d, batch_first=True,
                                         norm_first=True, dropout=0.0)
        self.tr = nn.TransformerEncoder(enc, nl)
        self.out = nn.Linear(d, 6)
    def forward(self, R, x, t):
        f = torch.cat([R.reshape(*R.shape[:2], 9), x], -1)
        h = self.inp(f) + self.pos(torch.arange(L, device=x.device)) \
            + self.tmlp(t[:, None])[:, None, :]
        o = self.out(self.tr(h))
        return o[..., :3], o[..., 3:]

model = FrameNet()
print('parameters:', sum(p.numel() for p in model.parameters()))

### Rep 4 — `se3_fm_loss(model, Rd, xd)`
Draw noise frames (`rand_rot` for rotations, `randn` for translations), draw
`t ~ U(0,1)` per backbone, build the point on the geodesic, and regress both
velocities with MSE.

Watch the broadcasting: `t` is `[n]`, while `w` is `[n, L, 3]`, so index it as
`t[:, None, None]`.

In [ ]:
def se3_fm_loss(model, Rd, xd):
    '''Conditional flow matching on SE(3): rotation geodesic + translation line.'''
    # YOUR CODE HERE
    # hint: n = len(Rd); Rn = rand_rot(n * L).reshape(n, L, 3, 3); xn = torch.randn_like(xd)
    #       w = so3_log(Rd.transpose(-1, -2) @ Rn);  v = xn - xd
    #       Rt = Rd @ so3_exp(t[:, None, None] * w)
    #       return MSE(w_pred, w) + MSE(v_pred, v)
    raise NotImplementedError

# --- checkpoint ---
l = se3_fm_loss(model, R_data[:32], X_data[:32])
assert l.dim() == 0 and l.requires_grad, 'need a differentiable scalar'
assert 0.5 < l.item() < 10.0, 'untrained loss should be order 1-10'
print('untrained SE(3) flow-matching loss %.3f ✓' % l.item())

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
hist, t0 = [], time.time()
for step in range(1, 1501):
    ix = torch.randint(0, len(R_data), (64,))
    loss = se3_fm_loss(model, R_data[ix], X_data[ix])
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0 or step == 1:
        hist.append((step, loss.item()))
        if step % 500 == 0 or step == 1:
            print('step %4d  loss %.4f' % (step, loss.item()))
print('trained in %.1fs on CPU' % (time.time() - t0))

h = np.array(hist)
fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.plot(h[:, 0], h[:, 1], color=BLUE, lw=2)
ax.set_xlabel('step'); ax.set_ylabel('MSE (rotation + translation)')
ax.set_title('regressing velocities on SE(3)'); ax.grid(alpha=.15)
plt.show()

### Rep 5 — `se3_sample(model, n, steps=20)`
Integrate the ODE from noise (`t=1`) to data (`t=0`). The translation update is
identical to 1.4. The rotation update is the one line that makes this notebook:

```
R  <-  R @ so3_exp(-dt * w_pred)
```

**Multiply by an exponential; never add.** Because `so3_exp` returns a rotation and the
product of two rotations is a rotation, `R` is *algebraically guaranteed* to stay on the
manifold for as many steps as you like. The checkpoint verifies exactly that.

In [ ]:
@torch.no_grad()
def se3_sample(model, n, steps=20):
    '''Generate n backbones by integrating the learned velocity field on SE(3).'''
    # YOUR CODE HERE
    # hint: R = rand_rot(n * L).reshape(n, L, 3, 3); x = torch.randn(n, L, 3)
    #       dt = 1.0 / steps; for i in range(steps): t = 1.0 - i * dt
    #       wp, vp = model(R, x, torch.full((n,), t))
    #       R = R @ so3_exp(-dt * wp);  x = x - dt * vp
    raise NotImplementedError

# --- checkpoint ---
Rg, xg = se3_sample(model, 400, steps=50)
assert Rg.shape == (400, L, 3, 3) and xg.shape == (400, L, 3)
orth = (Rg @ Rg.transpose(-1, -2) - torch.eye(3)).abs().max().item()
assert orth < 1e-4, 'rotations drifted off SO(3) — are you multiplying by so3_exp?'
assert (torch.det(Rg) - 1).abs().max() < 1e-4, 'determinants must stay +1'
print('after 50 sampling steps: ||R R^T - I|| = %.1e, det error = %.1e' %
      (orth, (torch.det(Rg) - 1).abs().max()))
print('Still exactly on the manifold — no projection, no renormalisation needed ✓')

## Part 4 — did it learn secondary structure?

The test is not "does it look protein-ish." It is whether each generated backbone
**commits** to one of the two twist regimes instead of hedging in between — the same
mode-coverage question as 1.3, now on a manifold.

In [ ]:
def commitment(Rs):
    m = twist_angles(Rs).mean(1)
    return ((m - 100).abs() < 25).float().mean().item(), ((m - 180).abs() < 25).float().mean().item()

print('steps   -> helix-like   strand-like   (real data is 50/50)')
for st in (5, 20, 50):
    Rg_, _ = se3_sample(model, 400, steps=st)
    a, b = commitment(Rg_)
    print('%5d      %.2f          %.2f       (%.0f%% committed)' % (st, a, b, 100 * (a + b)))

Rg, xg = se3_sample(model, 600, steps=50)
fig, ax = plt.subplots(figsize=(5.8, 3.4))
bins = np.linspace(0, 200, 45)
ax.hist(twist_angles(R_data).mean(1).numpy(), bins=bins, color=GREEN, alpha=.55,
        label='real', edgecolor='white', lw=.4)
ax.hist(twist_angles(Rg).mean(1).numpy(), bins=bins, color=BLUE, alpha=.55,
        label='generated', edgecolor='white', lw=.4)
ax.set_xlabel('mean twist per residue (degrees)'); ax.set_ylabel('backbones')
ax.set_title('both modes reproduced, in the right proportion')
ax.grid(alpha=.15); ax.legend(frameon=False, fontsize=9)
plt.show()

fig = plt.figure(figsize=(13, 3.4))
order = twist_angles(Rg).mean(1).argsort()
for k, idx in enumerate([order[10], order[100], order[-100], order[-10]]):
    ax = fig.add_subplot(1, 4, k + 1, projection='3d')
    show3d(ax, xg[idx], 'generated (twist %.0f deg)' % twist_angles(Rg).mean(1)[idx])
plt.tight_layout(); plt.show()
print('Generated backbones, built entirely from noise on SE(3). ✓')

## Reflection — what just transferred

- **Constrained data needs constrained arithmetic.** Rotations are not nine free
  numbers. Add noise to them and you leave the manifold immediately; multiply by
  `so3_exp` and you never can. That single change is what makes structure generation
  work at all.
- **The tangent space is where the machine learning happens.** `log` maps the manifold
  into a flat vector space, you do ordinary regression there, and `exp` maps back. The
  network never has to know it is on a manifold.
- **Flow matching transferred with one substitution.** Straight line becomes geodesic;
  the velocity is still constant along the path; the loss is still MSE. Curved-space
  generative modelling was mostly a change of two functions.
- **Frames are the right object** — the lesson 0.1 promised. Generate `(R, t)` per
  residue and you are generating a *pose*, not a coordinate soup.
- **What real systems add** (worth knowing before you read RFdiffusion): proper SE(3)
  equivariance so the model does not depend on global orientation, IGSO(3) noise for
  the diffusion formulation, a schedule that treats rotations and translations at
  different rates, and sequence conditioning. The core you just wrote is the same.

**Track 1 is complete.** You now have five generative engines: autoregressive, masked,
diffusion, flow, and discrete diffusion — plus this one for structure. Every one of
them samples `p(x)`, and *none* of them yet takes an instruction.

**Next rep:** `2.1 Conditional generation & classifier-free guidance` — Track 2 starts,
and this is where the whole gym has been heading. Stop sampling `p(x)` and start
sampling `p(x | y)`: tell the model what you want, and turn a knob for how insistently
it should listen.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def so3_exp(w):
    th = w.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    K = hat(w / th)
    th = th[..., None]
    I = torch.eye(3, dtype=w.dtype).expand_as(K)
    return I + torch.sin(th) * K + (1 - torch.cos(th)) * (K @ K)

def so3_log(R):
    tr = R[..., 0, 0] + R[..., 1, 1] + R[..., 2, 2]
    cos = ((tr - 1) / 2).clamp(-1 + 1e-7, 1 - 1e-7)
    th = torch.acos(cos)
    v = torch.stack([R[..., 2, 1] - R[..., 1, 2],
                     R[..., 0, 2] - R[..., 2, 0],
                     R[..., 1, 0] - R[..., 0, 1]], -1)
    return v * (th / (2 * torch.sin(th).clamp_min(1e-7)))[..., None]

def twist_angles(Rs):
    rel = Rs[:, :-1].transpose(-1, -2) @ Rs[:, 1:]
    return so3_log(rel).norm(dim=-1) * 180 / math.pi

def se3_fm_loss(model, Rd, xd):
    n = len(Rd)
    Rn = rand_rot(n * L).reshape(n, L, 3, 3)
    xn = torch.randn_like(xd)
    t = torch.rand(n)
    w = so3_log(Rd.transpose(-1, -2) @ Rn)          # constant along the geodesic
    v = xn - xd
    Rt = Rd @ so3_exp(t[:, None, None] * w)
    xt = (1 - t[:, None, None]) * xd + t[:, None, None] * xn
    wp, vp = model(Rt, xt, t)
    return F.mse_loss(wp, w) + F.mse_loss(vp, v)

@torch.no_grad()
def se3_sample(model, n, steps=20):
    R = rand_rot(n * L).reshape(n, L, 3, 3)
    x = torch.randn(n, L, 3)
    dt = 1.0 / steps
    for i in range(steps):
        t = 1.0 - i * dt
        wp, vp = model(R, x, torch.full((n,), t))
        R = R @ so3_exp(-dt * wp)                   # MULTIPLY: stays on SO(3) forever
        x = x - dt * vp
    return R, x

print('reference solutions loaded — re-run the checkpoint cells above')